# Walk Forward Validation for Strategy Backtesting

### Introduction

**Walk Forward Validation** (also known as Walk Forward Analysis or Rolling Window Out-of-Sample testing) is a robust technique used in quantitative trading and algorithmic strategy development to evaluate the true out-of-sample performance of a trading strategy.

Unlike traditional backtesting, which risks severe overfitting by optimizing parameters on the entire historical dataset, walk forward validation mimics real-world trading conditions more realistically:

- The data is divided into multiple **in-sample (training)** and **out-of-sample (testing)** periods.
- The strategy is optimized on the in-sample window.
- Performance is then evaluated on the subsequent unseen out-of-sample window.
- The windows are then "walked forward" through time, repeating the process.

This approach provides a much more realistic estimate of how a strategy might perform in live trading by reducing curve-fitting and ensuring the strategy is repeatedly tested on truly unseen future data.

In this notebook, we will:
- Implement walk forward validation from scratch
- Apply it to a sample trading strategy
- Analyze the stability and robustness of strategy parameters
- Compare results with traditional full-period backtesting along with the benchmark

### Step 1: Load the libraries

Import the necessary libraries

In [45]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import plotly.graph_objects as go

from datetime import datetime
from matplotlib import pyplot as plt
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

### Step 2: Import our data

We will use **yfinance** to download historical price data directly from Yahoo Finance. In this example, we load daily OHLCV data for **SPY** (S&P 500 ETF) over the last 15+ years. You can easily change the ticker or time period as needed.

In [2]:
stock_ticker = "AAPL"
benchmark_ticker = "SPY"
start_date = "2010-01-01"
end_date = "2025-12-31"

In [ ]:
stock_data = yf.download(stock_ticker, start=start_date, end=end_date)['Close']
benchmark_data = yf.download(benchmark_ticker, start=start_date, end=end_date)['Close']

# Rename the columns for clarity
stock_data.rename(columns={stock_ticker: "Close"}, inplace=True)
benchmark_data.rename(columns={benchmark_ticker: "Close"}, inplace=True)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


### Step 3: Devise Our Systematic Strategy

In this section, we will be implementing a classic **Dual Moving Average Crossover** momentum strategy. This is a straight forward systematic strategy that we will be using to illustrate the concept of WFA.

**Strategy Rules**
- **Long Entry**: When the short-term Simple Moving Average (SMA) crosses **above** the long-term SMA  
- **Short Entry**: When the short-term SMA crosses **below** the long-term SMA  

This is a trend-following strategy that aims to capture sustained directional moves in the market.

**Parameters**
- **Short-term SMA** → Fast moving average (default: 50 days)
- **Long-term SMA** → Slow moving average (default: 200 days)

We will generate trading signals based on these crossovers and later evaluate the strategy using Walk Forward Validation.

In [4]:
def simple_moving_average_cross_over(data, short_window, long_window):
  df = data.copy()
  df[f'SMA_{short_window}'] = df['Close'].rolling(window=short_window).mean()
  df[f'SMA_{long_window}'] = df['Close'].rolling(window=long_window).mean()
  
  # Devise our trading signals
  df['Position'] = 0
  df['Position'] = np.where(df[f'SMA_{short_window}'] > df[f'SMA_{long_window}'], 1, -1)
  df['Signal'] = df['Position'].diff()
  
  # Compute strategy returns
  df['Strategy_Returns'] = df['Close'].pct_change() * df['Position'].shift(1) # Shift by 1 to align with the next day's returns and avoid look-ahead bias
  return df

def hyperparameter_ma_tuning(data, short_window_range, long_window_range):
  best_sharpe = -np.inf
  best_params = None
  best_strategy = None
  short_window_lst = []
  long_window_lst = []
  sharpe_ratios_lst = []
  
  hyperparam_results = pd.DataFrame(columns=['Short_Window', 'Long_Window', 'Sharpe_Ratio'])
  
  for short_window in short_window_range:
    for long_window in long_window_range:
      if short_window >= long_window:
        continue  # Ensure short window is less than long window
      
      strategy_df = simple_moving_average_cross_over(data, short_window, long_window)
      sharpe_ratio = strategy_df['Strategy_Returns'].mean() / strategy_df['Strategy_Returns'].std() * np.sqrt(252) # Annualized Sharpe Ratio
      
      short_window_lst.append(short_window)
      long_window_lst.append(long_window)
      sharpe_ratios_lst.append(sharpe_ratio)
      
      if sharpe_ratio > best_sharpe:
        best_sharpe = sharpe_ratio
        best_params = (short_window, long_window)
        best_strategy = strategy_df
  
  hyperparam_results['Short_Window'] = short_window_lst
  hyperparam_results['Long_Window'] = long_window_lst
  hyperparam_results['Sharpe_Ratio'] = sharpe_ratios_lst
  
  return best_params, best_sharpe, best_strategy, hyperparam_results

### Step 4: Test the Systematic Strategy on Train and Test Data

Before applying the **Walk Forward Validation** technique, let us first understand the basic performance of our Dual Moving Average Crossover strategy using a simple **train/test split**.

We will:
- Split the data chronologically into a training set (in-sample) and a test set (out-of-sample)
- Optimize or define the strategy parameters on the **train** period only
- Evaluate the strategy performance on the unseen **test** period

This single split approach is a basic form of out-of-sample testing, but it still has limitations (e.g., results can be sensitive to the exact split date). Walk Forward Validation in the next steps will address this.

In [9]:
cutoff_date = '2019-01-01'
train_data, test_data = stock_data[:cutoff_date], stock_data[cutoff_date:]
short_window_range, long_window_range = range(10, 50, 2), range(50, 200, 5)
best_params, best_sharpe, best_strategy, hyperparam_results = hyperparameter_ma_tuning(train_data, short_window_range, long_window_range)

Visualise the Heatmap for Hypeparameter Tuning

In [24]:
def plot_hyperparameter_surface(hyperparam_results, best_params, three_dim_plot=True):
  # 3D Surface Plot of Hyperparameter Tuning Results
  if three_dim_plot:
    pivot_table = hyperparam_results.pivot(index='Short_Window', columns='Long_Window', values='Sharpe_Ratio')
    fig = go.Figure(data=go.Surface(z=pivot_table.values,x=pivot_table.columns,y=pivot_table.index,colorscale='Viridis',colorbar=dict(title='Sharpe Ratio')))

    # Mark the best point in 3D
    fig.add_trace(
      go.Scatter3d(
        x=[best_params[1]],
        y=[best_params[0]],
        z=[hyperparam_results['Sharpe_Ratio'].max() + 0.05],
        mode='markers+text',
        marker=dict(color='red', size=8, symbol='diamond'),
        text=[f"Best: ({best_params[0]}, {best_params[1]})"],textposition="top center",name='Best Hyperparameters')
      )
    fig.update_layout(
        title='3D Surface: Sharpe Ratio vs Short & Long Windows',
        scene=dict(xaxis_title='Long Window', yaxis_title='Short Window', zaxis_title='Sharpe Ratio', camera=dict(eye=dict(x=1.8, y=1.8, z=1.2))),
        height=800,
        margin=dict(l=10, r=10, b=10, t=50, pad=0),
        template='plotly_white'
    )
    fig.show()
  # 2D Heatmap of Hyperparameter Tuning Results
  else:
    # Plotly heatmap of hyperparameter tuning results
    pivot_table = hyperparam_results.pivot(index='Short_Window', columns='Long_Window', values='Sharpe_Ratio')
    fig = go.Figure(data=go.Heatmap(z=pivot_table.values,x=pivot_table.columns,y=pivot_table.index,colorscale='Viridis',colorbar=dict(title='Sharpe Ratio')))
    # Circle the best hyperparameters
    fig.add_trace(go.Scatter(
        x=[best_params[1]],
        y=[best_params[0]],
        mode='markers',
        marker=dict(color='red', size=30, symbol='circle-open', line=dict(color='red', width=2)),
        name='Best Hyperparameters'
    ))
    fig.update_layout(title='Hyperparameter Tuning Results (Sharpe Ratio)', xaxis_title='Long Window', yaxis_title='Short Window')
    fig.show()

In [25]:
plot_hyperparameter_surface(hyperparam_results, best_params, three_dim_plot=True)

### Step 5: Comparison of Strategy Performance Against Benchmark on Test Data

Now that we have tested our momentum strategy with both default parameters (50/200) and the hypertuned parameters, we will compare their performance against the **Buy & Hold** benchmark on the out-of-sample test period.

This comparison helps us evaluate whether the moving average crossover strategy (in either version) adds value over simply holding the asset.

In [ ]:
def plot_strategy_returns(strategy_data, benchmark_data, sample_name):
  performance_comparison = strategy_data.copy()
  performance_comparison['Benchmark_Returns'] = benchmark_data.pct_change()
  performance_comparison['Cumulative_Strategy_Returns'] = (1 + performance_comparison['Strategy_Returns']).cumprod() - 1
  performance_comparison['Cumulative_Benchmark_Returns'] = (1 + performance_comparison['Benchmark_Returns']).cumprod() - 1

  # Plot the cumulative returns of the strategy vs the benchmark
  fig = go.Figure()
    
  fig.add_trace(go.Scatter(
    x=performance_comparison.index,
    y=performance_comparison['Cumulative_Strategy_Returns'],
    mode='lines',
    name='Strategy',
    line=dict(color='blue', width=2.5)
  ))
  
  fig.add_trace(go.Scatter(
    x=performance_comparison.index,
    y=performance_comparison['Cumulative_Benchmark_Returns'],
    mode='lines',
    name='Benchmark (Buy & Hold)',
    line=dict(color='black', width=2, dash='dot')
  ))
  
  fig.update_layout(
    title=f'Simple Backtest Performance on {sample_name} Data<br>'
          '<span style="font-size:13px">(Normalized Cumulative Returns - Starting at 1.0)</span>',
    xaxis_title='Date',
    yaxis_title='Normalized Cumulative Return',
    height=620,
    template='plotly_white',
    
    # Overlay legend (top-left)
    legend=dict(
      x=0.02,
      y=0.98,
      xanchor='left',
      yanchor='top',
      bgcolor='rgba(255, 255, 255, 0.85)',   # semi-transparent white
      bordercolor='gray',
      borderwidth=1,
      font=dict(size=12)
    ),
    
    margin=dict(l=40, r=30, t=80, b=60)
  )
  fig.show()

def performance_metrics(strategy_data, benchmark_data):
  performance_comparison = strategy_data.copy()
  performance_comparison['Benchmark_Returns'] = benchmark_data.pct_change()
  performance_comparison['Cumulative_Strategy_Returns'] = (1 + performance_comparison['Strategy_Returns']).cumprod() - 1
  performance_comparison['Cumulative_Benchmark_Returns'] = (1 + performance_comparison['Benchmark_Returns']).cumprod() - 1

  metrics = pd.DataFrame({
    'Metric': ['Sharpe Ratio', 'Total Returns (%)', 'Annualized Volatility (%)'],
    'Strategy': [
      performance_comparison['Strategy_Returns'].mean() / performance_comparison['Strategy_Returns'].std() * np.sqrt(252),
      performance_comparison['Cumulative_Strategy_Returns'][-1] * 100,
      performance_comparison['Strategy_Returns'].std() * np.sqrt(252) * 100
    ],
    'Benchmark': [
      performance_comparison['Benchmark_Returns'].mean() / performance_comparison['Benchmark_Returns'].std() * np.sqrt(252),
      performance_comparison['Cumulative_Benchmark_Returns'][-1] * 100,
      performance_comparison['Benchmark_Returns'].std() * np.sqrt(252) * 100
    ]
  })
  return metrics

In [103]:
train_strategy = best_strategy.copy()
test_strategy = simple_moving_average_cross_over(test_data, best_params[0], best_params[1])

In [104]:
plot_strategy_returns(best_strategy, benchmark_data, "Train")
performance_metrics(best_strategy, benchmark_data)

,Metric,Strategy,Benchmark
0,Sharpe Ratio,0.912709,0.800643
1,Total Returns (%),510.803240,164.338390
2,Annualized Volatility (%),25.693576,14.914543


In [105]:
plot_strategy_returns(test_strategy, benchmark_data, "Test")
performance_metrics(test_strategy, benchmark_data)

,Metric,Strategy,Benchmark
0,Sharpe Ratio,0.166973,0.908682
1,Total Returns (%),2.327194,205.661828
2,Annualized Volatility (%),31.154160,19.778998


### Step 6: Live / Out-of-Sample Performance of Hypertuned Parameters

As seen above, our hypertuned parameters failed to outperform the benchmark on the live data using the simple backtest strategy.

**Analysis**

While the optimized short and long windows delivered strong performance during the in-sample phase they underperformed the simple Buy & Hold benchmark when tested on the most recent out-of-sample (live) period. This is a common and important observation in quantitative strategy development. It highlights several key lessons:
- **Overfitting risk**: When we simply perform hyperparameter tuning on our training set, we are essentially overfitting.
- **Regime change**: Market conditions in the live period may have differed significantly from the optimization window (e.g., strong bull market, low volatility, or different trend behavior).
- **Parameter instability**: The “best” parameters found during tuning may not be robust across all market environments.

**Next Steps**

**Key Observations**

The hypertuned parameters, despite showing strong results during optimization, failed to outperform the Buy & Hold benchmark on the recent out-of-sample (live) data. This is a classic symptom in strategy development and highlights the limitations of single train/test splits or traditional hyperparameter tuning. **This result strongly motivates the need for more rigorous robustness testing.**

In the next section, we will implement **Walk Forward Validation** — a more realistic and industry-standard approach that repeatedly trains and tests the strategy across rolling windows of time. This method provides a far better estimate of how the strategy might perform in real trading conditions by reducing overfitting and better simulating live deployment.

### Step 7: Implementing the Walk-Forward Validation

Walk Forward Validation is one of the most robust methods for evaluating trading strategies. It simulates real-world trading by repeatedly optimizing the strategy on an **in-sample** (training) period and then testing it on a subsequent unseen **out-of-sample** period. The windows then roll forward through time. This approach significantly reduces overfitting compared to traditional backtesting or a single train/test split.

Below is a visual representation of the Walk Forward process: 

![Walk Foward Validation Sample](https://miro.medium.com/0*MePEKgLfIGULgrdi)

#### Walk-Forward Setup

Given our dataset spanning 2010–2025, we configure the walk-forward analysis as follows:
- **In-Sample Period**: 4 years (used for optimizing strategy parameters)
- **Out-of-Sample Period**: 1 year (used for testing the optimized parameters)
- **Step Size**: 1 year (how much the windows move forward each iteration)

This creates a rolling 5-years cycle (4 years IS + 1 year OOS) with a 1-year step.

#### Implementation

We will now implement this walk-forward framework from scratch. In each iteration:
1. Optimize the short and long SMA windows on the current in-sample data.
2. Apply the best parameters to the following out-of-sample period.
3. Record the out-of-sample performance.
4. Roll the windows forward by the step size and repeat.

In [42]:
def walk_forward_validation(data, short_window_range, long_window_range, insample_range, outsample_range, step_size):
  """
    Performs walk-forward validation on the given data.
    
    Parameters:
    - data: pandas DataFrame with OHLCV data (must have datetime index)
    - short_window_range: range of short moving average windows to test
    - long_window_range: range of long moving average windows to test
    - insample_range: number of bars for in-sample (training + warmup) period
    - outsample_range: number of bars for out-of-sample (validation/testing) period
    - step_size: how many bars to step forward each time
    
    Returns:
    - DataFrame of backtest results for each out-of-sample period concatenated together for analysis
  """
  outsample_performance = []
  total_window = insample_range + outsample_range # Calculate the total window size (insample + outsample)  
  min_start_idx = total_window # Find the starting index where we have enough data for the first full window
    
  for i in range(min_start_idx, len(data) - outsample_range + 1, step_size):
    # Define the in-sample (training) period
    train_end = i
    train_start = train_end - insample_range
    data_train = data.iloc[train_start:train_end]
    
    # Define the out-of-sample (validation) period
    test_start = train_end
    test_end = test_start + outsample_range
    data_test = data.iloc[test_start:test_end]
    
    # Hyperparameter tuning on the training data
    best_params, best_sharpe, best_strategy, hyperparam_results = hyperparameter_ma_tuning(data_train, short_window_range, long_window_range)
    
    # Backtest on the out-of-sample data using the best hyperparameters
    data_test = simple_moving_average_cross_over(data_test, best_params[0], best_params[1])    
    
    # Add the best params pair to the outsample performance results for later analysis
    test_performance = pd.DataFrame() # Placeholder for actual performance metrics calculation
    test_performance['Date'] = data_test.index
    test_performance['Strategy Returns'] = list(data_test['Strategy_Returns'])
    test_performance['Best_Short_Window'] = best_params[0]
    test_performance['Best_Long_Window'] = best_params[1]
    outsample_performance.append(test_performance)  # placeholder - replace with stats_validation
  
  # Concat the outsample performance results into a single DataFrame for easier analysis
  outsample_performance = pd.concat(outsample_performance).reset_index(drop=True)
  return outsample_performance

Run the walk-forward validation

In [52]:
short_window_range = range(10, 50, 2)
long_window_range = range(50, 200, 5)  
insample_range = 252 * 4  # 4 years of daily data for training
outsample_range = 252     # 1 year of daily data for testing
step_size = 252           # Step forward by 1 year

walk_forward_validation_results = walk_forward_validation(
  stock_data, 
  short_window_range=short_window_range, 
  long_window_range=long_window_range, 
  insample_range=insample_range, 
  outsample_range=outsample_range, 
  step_size=step_size
)

### Step 8: Visualise the Walk-Forward Validation Performance (during the WF period)

Now that we have run the Walk-Forward Validation, we can visualise how the strategy performed across all the out-of-sample periods.

This visualisation is very important because it shows:
- The equity curve built only from **true out-of-sample** results (no look-ahead bias)
- How consistent the strategy was across different market regimes
- Whether performance deteriorated over time

We will plot:
1. The combined Out-of-Sample equity curve (Walk-Forward equity)
2. Individual Out-of-Sample periods highlighted
3. Comparison against benchmark over the same combined OOS periods

In [53]:
# Compute summary statistics for each out-of-sample period
summary_stats = walk_forward_validation_results.groupby(['Best_Short_Window', 'Best_Long_Window']).agg(
    Total_Returns=('Strategy Returns', lambda x: (1 + x).prod() - 1),
    Annualized_Returns=('Strategy Returns', lambda x: (1 + x).prod() ** (252 / len(x)) - 1),
    Annualized_Volatility=('Strategy Returns', lambda x: x.std() * np.sqrt(252)),
    Sharpe_Ratio=('Strategy Returns', lambda x: (x.mean() / x.std()) * np.sqrt(252))
).reset_index()
summary_stats

,Best_Short_Window,Best_Long_Window,Total_Returns,Annualized_Returns,Annualized_Volatility,Sharpe_Ratio
0,12,60,-0.076290,-0.076290,0.358984,-0.042391
1,12,160,-0.052480,-0.052480,0.267424,-0.069096
2,16,105,0.026645,0.026645,0.231836,0.229215
3,24,195,-0.235059,-0.235059,0.242209,-0.989144
4,26,185,0.089847,0.089847,0.249120,0.470920
5,32,175,-0.219192,-0.219192,0.178236,-1.303884
6,38,75,0.555962,0.555962,0.304995,1.608031
7,42,90,0.300406,0.300406,0.470909,0.795555
8,46,115,-0.464806,-0.464806,0.198480,-3.058611
9,48,105,0.001907,0.001907,0.226492,0.122383


Visualise the Walk-Forward Validation Performance

In [54]:
# Create a combined label for the x-axis
summary_stats['Combo'] = (summary_stats['Best_Short_Window'].astype(str) + ' / ' + summary_stats['Best_Long_Window'].astype(str))
fig = go.Figure()
fig.add_bar(
  x=summary_stats['Combo'],
  y=summary_stats['Sharpe_Ratio'],
  name='Sharpe Ratio',
  text=summary_stats['Sharpe_Ratio'].round(3),      # Show values on top of bars
  textposition='outside'
)

fig.update_layout(
  title='Sharpe Ratio for Each Hyperparameter Combination',
  xaxis_title='Short Window / Long Window',
  yaxis_title='Sharpe Ratio',
  height=600,
  width=1300,                    # Adjust width if you have many combinations
  bargap=0.2,                    # Space between bars
  xaxis_tickangle=-45            # Rotate x labels for better readability
)
fig.show()

In [100]:
def walk_forward_validation_performance(wf_results, strategy_data, benchmark_data):
  # Create a new dataframe for plotting and comparing the hyperparameter combination, WFV and benchmark performance
  # Since we tested our initial strategy on 2019 data onwards, we will use data from 2019 onwards for the WFV results as well to ensure a fair comparison with the benchmark
  performance = pd.DataFrame()
  backtested_strat = strategy_data.copy()
  benchmark = benchmark_data.copy()
  
  # Perform the indexation to align with the dates of the walk-forward validation results
  filtered_data_range = backtested_strat[wf_results['Date'].min():wf_results['Date'].max()]
  backtested_strat_filtered = backtested_strat[backtested_strat.index.isin(filtered_data_range.index)]
  benchmark_filtered = benchmark[benchmark.index.isin(filtered_data_range.index)]
  
  # Compute cumulative returns for the backtested strategy and the benchmark
  benchmark_filtered['Cumulative_Benchmark_Returns'] = (1 + benchmark_filtered.pct_change()).cumprod() - 1
  backtested_strat_filtered['Cumulative_Strategy_Returns'] = (1 + backtested_strat_filtered['Strategy_Returns']).cumprod() - 1
  
  performance['Cumulative_Strategy_Returns'] = list(backtested_strat_filtered['Cumulative_Strategy_Returns'])
  performance['Cumulative_Benchmark_Returns'] = list(benchmark_filtered['Cumulative_Benchmark_Returns'])

  # Filter the walk-forward validation results to the same date range for a fair comparison
  wf_results_filtered = wf_results[wf_results['Date'].between(filtered_data_range.index.min(), filtered_data_range.index.max())]
  performance['Walk-Forward Validation Strategy Returns'] = list((1 + wf_results_filtered['Strategy Returns']).cumprod() - 1)
  performance['Date'] = list(wf_results_filtered['Date'])

  # Convert to datetime and set index for plotting
  performance['Date'] = pd.to_datetime(performance['Date'])
  performance.set_index('Date', inplace=True)
  
  # Plot the cumulative returns of the WFV strategy vs the benchmark and the initial backtest strategy
  
  fig = go.Figure()  
  fig.add_trace(go.Scatter(
    x=performance.index, 
    y=performance['Walk-Forward Validation Strategy Returns'],
    name='Walk-Forward Validation (OOS)',
    line=dict(color='purple', width=2.8)
  ))
  
  fig.add_trace(go.Scatter(
    x=performance.index, 
    y=performance['Cumulative_Strategy_Returns'],
    name='Initial Backtest Strategy',
    line=dict(color='royalblue', width=2, dash='dash')
  ))
  
  fig.add_trace(go.Scatter(
    x=performance.index, 
    y=performance['Cumulative_Benchmark_Returns'],
    name='Benchmark (Buy & Hold)',
    line=dict(color='black', width=2, dash='dot')
  ))
  
  fig.update_layout(
      title='Walk-Forward Validation Performance vs Initial Backtest vs Benchmark<br>'
            '<span style="font-size:13px">(All series normalized to 1.0 starting from 2019)</span>',
      xaxis_title='Date',
      yaxis_title='Normalized Cumulative Return',
      height=620,
      template='plotly_white',
      
      # === Legend overlaid on the chart ===
      legend=dict(
        x=0.02,           # position: left side
        y=0.98,           # position: near the top
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',   # semi-transparent white background
        bordercolor='gray',
        borderwidth=1,
        font=dict(size=11)
      ),
      
      margin=dict(l=40, r=20, t=80, b=60)
  )
  fig.show()


def complete_performance_comparison(wf_results, strategy_data, benchmark_data):
  # This function can be used to compute and print performance metrics for the WFV strategy, the initial backtest strategy, and the benchmark for a comprehensive comparison
  # Performance evaluation of the WFV strategy vs the benchmark on the same out-of-sample periods
  # Quick formatted DataFrame (no extra function)
  backtested_strat = strategy_data.copy()
  benchmark = benchmark_data.copy()
  wf_results = wf_results.copy()
  
  # Perform the indexation to align with the dates of the walk-forward validation results
  filtered_data_range = backtested_strat[wf_results['Date'].min():wf_results['Date'].max()]
  backtested_strat_filtered = backtested_strat[backtested_strat.index.isin(filtered_data_range.index)]
  benchmark_filtered = benchmark[benchmark.index.isin(filtered_data_range.index)]
  benchmark_filtered['Benchmark_Returns'] = benchmark_filtered.pct_change()
  wf_results_filtered = wf_results[wf_results['Date'].between(filtered_data_range.index.min(), filtered_data_range.index.max())]
  
  metrics = {
    'Metric': ['Total Return (%)', 'Sharpe Ratio', 'Annualized Volatility (%)'],
    'Benchmark': [
      round(benchmark_filtered['Benchmark_Returns'].iloc[-1] * 100, 2),
      round(benchmark_filtered['Benchmark_Returns'].mean() / benchmark_filtered['Benchmark_Returns'].std() * np.sqrt(252), 3),
      round(benchmark_filtered['Benchmark_Returns'].std() * np.sqrt(252) * 100, 2)
    ],
    'Initial Strategy': [
      round(backtested_strat_filtered['Strategy_Returns'].iloc[-1] * 100, 2),
      round(backtested_strat_filtered['Strategy_Returns'].mean() / backtested_strat_filtered['Strategy_Returns'].std() * np.sqrt(252), 3),
      round(backtested_strat_filtered['Strategy_Returns'].std() * np.sqrt(252) * 100, 2)
    ],
    'Walk-Forward Strategy (OOS)': [
      round(wf_results_filtered['Strategy Returns'].iloc[-1] * 100, 2),
      round(wf_results_filtered['Strategy Returns'].mean() / wf_results_filtered['Strategy Returns'].std() * np.sqrt(252), 3),
      round(wf_results_filtered['Strategy Returns'].std() * np.sqrt(252) * 100, 2)
    ]
  }

  comparison_df = pd.DataFrame(metrics).set_index('Metric')
  return comparison_df       

Visualise the returns over time

In [101]:
walk_forward_validation_performance(walk_forward_validation_results, test_strategy, benchmark_data)
complete_performance_comparison(walk_forward_validation_results, test_strategy, benchmark_data)

,Benchmark,Initial Strategy,Walk-Forward Strategy (OOS)
Metric,,,
Total Return (%),-1.530,-2.410,-2.410
Sharpe Ratio,0.885,0.176,-0.153
Annualized Volatility (%),19.820,30.900,30.900


### Conclusion: Walk-Forward Validation – Strengths and Pitfalls

Walk-Forward Validation (also known as Walk-Forward Analysis or Optimization) is one of the most robust techniques available for evaluating trading strategies. By repeatedly optimizing parameters on an in-sample window and testing them on a subsequent unseen out-of-sample window — then rolling forward through time — it closely simulates real-world trading conditions and significantly reduces the risk of **overfitting** compared to traditional full-period backtesting or a single train/test split.

#### Key Advantages
- Provides a more realistic estimate of live trading performance by using only out-of-sample results.
- Tests the strategy’s adaptability across different market regimes.
- Validates not just the parameters, but also the optimization process itself.
- Helps reveal whether a strategy’s edge is genuine or merely curve-fitted to historical noise.

In our case, the Walk-Forward results offered a clearer (and often more conservative) picture than the initial backtest, highlighting where the Dual Moving Average strategy struggled in certain periods.

#### Important Pitfalls and Limitations

Despite its strengths, Walk-Forward Validation is not foolproof. Common pitfalls include:
- **Window Size and Step Selection Bias** — The choice of in-sample (e.g., 9 months) and out-of-sample periods (e.g., 3 months), as well as the step size, can heavily influence results. Poorly chosen windows may miss important market cycles or introduce seasonal biases.
- **Lag in Adapting to Regime Changes** — The method reacts to new market regimes rather than predicting them, so performance can degrade during sudden shifts (e.g., from trending to range-bound markets).
- **Residual Overfitting Risk** — If you test too many parameter combinations, optimization criteria (“fitness function shopping”), or window configurations, you can still overfit the walk-forward process itself.
- **High Computational Cost** — Multiple optimization runs across rolling windows make it significantly slower and more resource-intensive than simple backtesting.
- **Data Requirements** — It needs a long historical dataset to produce enough out-of-sample periods for statistically meaningful conclusions.
- **Parameter Instability** — Large swings in optimal parameters across windows often signal a non-robust strategy.

### Final Takeaway

In this notebook, our momentum strategy showed decent but not exceptional walk-forward performance — a realistic outcome that reminds us: even sophisticated validation methods cannot guarantee future profits. Markets evolve, and no backtesting framework can fully capture the uncertainty of live trading.